In [ ]:
#Libaray used for automatic EDA
!pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 30.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 30.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 7.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.7 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.46.0
    Uninstalling llvmlite-0.46.0:
      Successfully uninstalled llvmlite-0.46.0
  Attempting uninstall: numba
    Found existing installation: numba 0.64.0
    Uninstalling numba-0.64.0:
      Successfully uninstalled numba-0.64.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [ydata-profiling]


In [ ]:
!pip install --ignore-installed numpy==2.3.3

In [ ]:
# After restarting your session (Runtime > Restart session), run this cell to import the libraries.
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import warnings
import pickle
warnings.filterwarnings('ignore')

In [ ]:
books = pd.read_csv("Books.csv")
ratings = pd.read_csv("Ratings.csv")
users = pd.read_csv("Users.csv")



```
# This is formatted as code
```

### **Books Dataset Data Pre Processing**



In [ ]:
books.head(2)

In [ ]:
print(books.shape)
print("-----------------------------------------------------------------")
print("Null Values:" , books.isnull().sum())
print("-----------------------------------------------------------------")

print("Books Info: ", books.info())
print("-----------------------------------------------------------------")

print("Books describe that: ", books.describe())
print("-----------------------------------------------------------------")

print("Duplicated Books: " , books.duplicated().sum())
print("-----------------------------------------------------------------")



In [ ]:
# Convert Year-Of-Publication to numeric, coercing errors to NaN
years = pd.to_numeric(books['Year-Of-Publication'], errors='coerce')

# Filter out NaN values and focus on a reasonable range of years (e.g., 1950 to 2010)
years = years[(years >= 1950) & (years <= 2010)]

# Plot the distribution as a histogram
plt.figure(figsize=(10, 5))
plt.hist(years, bins=30, color='skyblue', edgecolor='black')
plt.title('Distribution of Books by Year of Publication (1950 - 2010)')
plt.xlabel('Year of Publication')
plt.ylabel('Number of Books')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Convert Year-Of-Publication to numeric, converting invalid strings like 'DK Publishing Inc' to NaN
years_cleaned = pd.to_numeric(books['Year-Of-Publication'], errors='coerce')

# Use modern sns.histplot with kernel density estimation (kde=True)
sns.distplot(years_cleaned, bins=30, hist=False)
plt.title('Distribution of Year of Publication')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()

### **Ratings Dataset Data Pre Processing**


In [ ]:
ratings.head(3)

In [ ]:
print(ratings.shape)
print("-----------------------------------------------------------------")
print("Null Values:" , ratings.isnull().sum())
print("-----------------------------------------------------------------")

print("Ratings Info: ", ratings.info())
print("-----------------------------------------------------------------")

print("Ratings describe that: ", ratings.describe())
print("-----------------------------------------------------------------")

print("Duplicated Ratings: " , ratings.duplicated().sum())
print("-----------------------------------------------------------------")


In [ ]:
sns.distplot(x=ratings['Book-Rating'])
plt.show()

### **Users Dataset Data Pre Processing**


In [ ]:
users.head()

In [ ]:
print(users.shape)
print("-----------------------------------------------------------------")
print("Null Values:" , users.isnull().sum())
print("-----------------------------------------------------------------")

print("Users Info: ", users.info())
print("-----------------------------------------------------------------")

print("Users describe that: ", users.describe())
print("-----------------------------------------------------------------")

print("Duplicated Users: " , users.duplicated().sum())
print("-----------------------------------------------------------------")


In [ ]:
# Convert Age to numeric, coercing errors to NaN
ages = pd.to_numeric(users['Age'], errors='coerce')

# Filter out NaN values and focus on a reasonable range of ages (e.g., 0 to 100)
ages = ages[(ages >= 0) & (ages <= 100)]

# Plot the distribution of user ages as a histogram
plt.figure(figsize=(10, 5))
plt.hist(ages, bins=10, color='skyblue', edgecolor='black')
plt.title('Distribution of User Ages')
plt.xlabel('Age')
plt.ylabel('Number of Users')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()



```
# This is formatted as code
```

# **Popularity based Book Recommender System**

In [ ]:
# Merging books and ratings on the top of ISBN Column
Books_with_ratings = ratings.merge(books , on ='ISBN')


In [ ]:
num_rating_df = Books_with_ratings.groupby('Book-Title').count()['Book-Rating'].reset_index()
num_rating_df.rename(columns={'Book-Rating':'num_ratings'},inplace=True)
num_rating_df

In [ ]:
avg_rating_df = Books_with_ratings.groupby('Book-Title')['Book-Rating'].mean().reset_index()
avg_rating_df.rename(columns={'Book-Rating':'avg_ratings'},inplace=True)
avg_rating_df

In [ ]:
popularity_df = num_rating_df.merge(avg_rating_df , on = 'Book-Title')
popularity_df.head()

In [ ]:
popularity_df = popularity_df[popularity_df['num_ratings']>=250].sort_values('avg_ratings',ascending=False).head(50)
popularity_df

In [ ]:
profile =ProfileReport(popularity_df, title="Pandas Profiling Report")
profile.to_file("output.html")

In [ ]:
popularity_df = popularity_df.merge(books, on='Book-Title').drop_duplicates('Book-Title')[['Book-Title', 'Book-Author', 'Image-URL-M', 'num_ratings', 'avg_ratings']]

In [ ]:
popularity_df



# **Collaborative Filtering based Book Recommender System**

In [ ]:
X = Books_with_ratings.groupby('User-ID').count()['Book-Rating'] > 200
cred_users = X[X].index

In [ ]:
filtered_ratings = Books_with_ratings[Books_with_ratings['User-ID'].isin(cred_users)]
filtered_ratings.head()

In [ ]:
Y = filtered_ratings.groupby('Book-Title').count()['Book-Rating']>=50
famous_books = Y[Y].index

In [ ]:
final_ratings = filtered_ratings[filtered_ratings['Book-Title'].isin(famous_books)]
final_ratings


In [ ]:
final_ratings.drop_duplicates()

In [ ]:
final_ratings.duplicated().sum()

In [ ]:
pt = final_ratings.pivot_table(index='Book-Title',columns='User-ID',values='Book-Rating')

In [ ]:
pt.fillna(0,inplace = True)

In [ ]:
pt

In [ ]:
similarity_score = cosine_similarity(pt)

In [ ]:
def recommend(book_name):
    # Find the index of the selected book in the pivot table
    index = np.where(pt.index == book_name)[0][0]

    # Get similarity scores, sort in descending order, and select top 5 similar books
    similar_items = sorted(
        list(enumerate(similarity_score[index])),
        key=lambda x: x[1],
        reverse=True
    )[1:6]  # Skip the first book because it is the book itself

    # Print the names of the 5 most similar books
    for i in similar_items:
        print(pt.index[i[0]])

In [ ]:
recommend('1984')

In [ ]:
pickle.dump(popularity_df, open('popular.pkl', 'wb'))